In [1]:
# https://pypi.org/project/pqcrypto/
!pip install pqcrypto cryptography


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
import sys
from secrets import compare_digest

# Import the NIST Level 3 HQC parameter set from the pqcrypto library
try:
    # from pqcrypto.kem.hqc_128 import generate_keypair, encrypt, decrypt
    # from pqcrypto.kem.hqc_192 import generate_keypair, encrypt, decrypt
    from pqcrypto.kem.hqc_256 import generate_keypair, encrypt, decrypt
except ImportError:
    print("[!] Error: 'pqcrypto' is not configured correctly or lacks HQC bindings.")
    sys.exit(1)

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

# ----------------------------------------------------------------------
# Symmetric Engine Layer (AES-256-GCM)
# ----------------------------------------------------------------------
def aes_gcm_encrypt(plaintext_message: bytes, symmetric_key: bytes) -> tuple[bytes, bytes, bytes]:
    """Encrypts arbitrary text using AES-256-GCM driven by the HQC shared secret."""
    # HQC-192 outputs a 64-byte raw shared secret. 
    # We slice the first 32 bytes to form a perfect cryptographically secure AES-256 key.
    aes_key = symmetric_key[:32]
    iv = os.urandom(12)  # 96-bit authenticated initialization vector
    
    encryptor = Cipher(
        algorithms.AES(aes_key),
        modes.GCM(iv),
        backend=default_backend()
    ).encryptor()
    
    ciphertext = encryptor.update(plaintext_message) + encryptor.finalize()
    return ciphertext, iv, encryptor.tag

def aes_gcm_decrypt(ciphertext: bytes, symmetric_key: bytes, iv: bytes, tag: bytes) -> bytes:
    """Decrypts ciphertext using the derived key and validates the authentication tag."""
    aes_key = symmetric_key[:32]
    decryptor = Cipher(
        algorithms.AES(aes_key),
        modes.GCM(iv, tag),
        backend=default_backend()
    ).decryptor()
    return decryptor.update(ciphertext) + decryptor.finalize()


# ----------------------------------------------------------------------
# HQC Core Execution Loop
# ----------------------------------------------------------------------
def main():
    print("=================================================================")
    print("            HQC (Hamming Quasi-Cyclic) + AES-GCM Demo            ")
    print("=================================================================\n")

    # Define our target data message
    target_message = b"We will serve the Lord our God and obey him."
    print(f"[+] Message to Encrypt: '{target_message.decode()}'")

    # 1. Asymmetric Key Generation (Receiver / Alice)
    print("\n[*] Alice is generating an HQC public/private key pair...")
    public_key, secret_key = generate_keypair()
    
    # Observe how the Quasi-Cyclic properties keep keys small compared to McEliece
    print(f"    -> Alice's Public Key Size : {len(public_key):,} bytes (~4.5 KB)")
    print(f"    -> Alice's Secret Key Size : {len(secret_key):,} bytes")

    # 2. Encapsulation & Symmetric Locking (Sender / Bob)
    print("\n[*] Bob is executing KEM Encapsulation and Data Encryption...")
    
    # Bob inputs Alice's public key matrix vector. The KEM outputs:
    # - hqc_ciphertext: The wrapped package to send over the network
    # - shared_secret_bob: The random plaintext key generated by the KEM math
    hqc_ciphertext, shared_secret_bob = encrypt(public_key)
    
    # Bob immediately uses this shared secret to lock the actual message payload via AES
    aes_ciphertext, iv, auth_tag = aes_gcm_encrypt(target_message, shared_secret_bob)

    print(f"    -> HQC Ciphertext Size     : {len(hqc_ciphertext):,} bytes (~4.5 KB)")
    print(f"    -> AES Ciphertext Size     : {len(aes_ciphertext)} bytes")
    print(f"    -> Derived Shared Key (Hex): {shared_secret_bob[:24].hex()}...")

    # ------------------------------------------------------------------
    # TRANSMISSION SIMULATION
    # Bob sends: (hqc_ciphertext, aes_ciphertext, iv, auth_tag)
    # ------------------------------------------------------------------
    print("\n" + "-"*60 + "\n[*] Transmitting packets to Alice...")

    # 3. Decapsulation & Decryption (Receiver / Alice)
    print("\n[*] Alice received the payload. Reversing the pipeline...")
    
    # Alice runs the HQC decoder using her secret key to remove the injected noise
    shared_secret_alice = decrypt(secret_key, hqc_ciphertext)
    
    # Ensure both sides derived the identical key in constant time
    if not compare_digest(shared_secret_bob, shared_secret_alice):
        raise ValueError("[🛑] Critical Security Failure: Shared secrets do not align.")
    
    print("    [✅] HQC KEM Keys validated successfully.")

    # Alice decrypts the symmetric block using the verified key
    decrypted_message = aes_gcm_decrypt(aes_ciphertext, shared_secret_alice, iv, auth_tag)
    
    print(f"    -> Recovered Message       : '{decrypted_message.decode()}'")

    # Final assertion safety check
    assert compare_digest(target_message, decrypted_message)
    print("\n[🎉] SUCCESS: Code-based HQC channel execution complete.")

if __name__ == "__main__":
    main()

            HQC (Hamming Quasi-Cyclic) + AES-GCM Demo            

[+] Message to Encrypt: 'We will serve the Lord our God and obey him.'

[*] Alice is generating an HQC public/private key pair...
    -> Alice's Public Key Size : 7,245 bytes (~4.5 KB)
    -> Alice's Secret Key Size : 7,317 bytes

[*] Bob is executing KEM Encapsulation and Data Encryption...
    -> HQC Ciphertext Size     : 14,421 bytes (~4.5 KB)
    -> AES Ciphertext Size     : 44 bytes
    -> Derived Shared Key (Hex): b783a4a11ece2036be5ee0580c8972f3b549475366fafda3...

------------------------------------------------------------
[*] Transmitting packets to Alice...

[*] Alice received the payload. Reversing the pipeline...
    [✅] HQC KEM Keys validated successfully.
    -> Recovered Message       : 'We will serve the Lord our God and obey him.'

[🎉] SUCCESS: Code-based HQC channel execution complete.
